# SafeCart — Product Image Collection Pipeline
### Sociolla direct extraction + rate-limited Serper.dev fallback

This notebook collects product packaging images for the SafeCart genuine/counterfeit
detection dataset.

**Strategy per product:**
1. Try direct image extraction from the product's Sociolla page.
2. If not enough images are found, fall back to **one** Serper Image Search request.
3. Serper usage is hard-capped at **2,500 requests total**, is thread-safe, persists
   across restarts, and never crashes the pipeline when the quota runs out — it simply
   disables the fallback and the rest of the pipeline (Sociolla extraction, local image
   processing, validation, dedup, metadata writing) keeps running.

> Genuine/counterfeit evidence logic is untouched by this notebook. This notebook
> focuses purely on image *collection*.


## 0. Setup — Imports & Paths

In [1]:
import os
import io
import csv
import json
import time
import hashlib
import random
import threading
from pathlib import Path
from datetime import datetime, timezone
from threading import Lock
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from PIL import Image

# ---- Project paths --------------------------------------------------------
PROJECT_ROOT = Path("SafeCart_Dataset")
IMAGES_DIR = PROJECT_ROOT / "images"
METADATA_DIR = PROJECT_ROOT / "metadata"
LOGS_DIR = PROJECT_ROOT / "logs"

for d in (PROJECT_ROOT, IMAGES_DIR, METADATA_DIR, LOGS_DIR):
    d.mkdir(parents=True, exist_ok=True)

SERPER_USAGE_FILE = METADATA_DIR / "serper_usage.json"
MISSING_IMAGES_FILE = METADATA_DIR / "missing_images.csv"
PRODUCTS_CSV = "products_sampe.csv"

print("Project root: ", PROJECT_ROOT.resolve())
print("Images dir:   ", IMAGES_DIR.resolve())
print("Metadata dir: ", METADATA_DIR.resolve())


Project root:  D:\COMPFEST\SOCIOLLA DATASET\2. Sociolla Real Image Product\SafeCart_Dataset
Images dir:    D:\COMPFEST\SOCIOLLA DATASET\2. Sociolla Real Image Product\SafeCart_Dataset\images
Metadata dir:  D:\COMPFEST\SOCIOLLA DATASET\2. Sociolla Real Image Product\SafeCart_Dataset\metadata


## 1. API Key Configuration

Paste your own Serper API key below. **Do not commit this notebook with a real key
pasted in** — the key is only ever kept in memory / the process environment variable
and is never written to disk, printed, or logged.

In [ ]:
# ============================================================
# 1. API KEY  —
# ============================================================
SERPER_API_KEY = ""  

if not SERPER_API_KEY.strip():
    raise ValueError(
        "SERPER_API_KEY is empty. Please paste your Serper API key."
    )

os.environ["SERPER_API_KEY"] = SERPER_API_KEY.strip()

# Never print the key itself — only confirm it loaded.
print("Serper API key loaded:", "YES" if os.environ.get("SERPER_API_KEY") else "NO")

# Free the plaintext variable from notebook display state as an extra precaution.
del SERPER_API_KEY


Serper API key loaded: YES


## 2. Request Limit, Counter & Thread Safety

In [3]:
# ============================================================
# 2. REQUEST LIMIT
# ============================================================
SERPER_MAX_REQUESTS = 2500
SERPER_ENABLED = True

# Thread-safe global counter (see Section 4)
serper_requests_used = 0
serper_lock = Lock()

MIN_IMAGES_PER_PRODUCT = 3          # threshold that triggers Serper fallback
SERPER_RETRY_LIMIT = 2              # retries for temporary network errors
SERPER_REQUEST_TIMEOUT = 15         # seconds
SERPER_SAVE_EVERY_N_PRODUCTS = 25   # persist usage checkpoint cadence

print("SERPER_MAX_REQUESTS  :", SERPER_MAX_REQUESTS)
print("MIN_IMAGES_PER_PRODUCT:", MIN_IMAGES_PER_PRODUCT)


SERPER_MAX_REQUESTS  : 2500
MIN_IMAGES_PER_PRODUCT: 3


## 3 & 7 & 8. Persistent, Resumable Usage Counter

Loads `serper_usage.json` if it already exists so restarting the notebook can never
push total usage past 2,500. Saved periodically (every 25 products) and after every
Serper request.

In [4]:
# ============================================================
# 7. PERSISTENT USAGE COUNTER  /  8. RESUME SUPPORT
# ============================================================

def load_serper_usage():
    """Load persisted Serper usage counter, if present, and sync globals."""
    global serper_requests_used, SERPER_ENABLED

    if SERPER_USAGE_FILE.exists():
        try:
            with open(SERPER_USAGE_FILE, "r") as f:
                data = json.load(f)
            with serper_lock:
                serper_requests_used = int(data.get("used", 0))
                if serper_requests_used >= SERPER_MAX_REQUESTS:
                    SERPER_ENABLED = False
            print(f"Resumed Serper usage from checkpoint: "
                  f"{serper_requests_used} / {SERPER_MAX_REQUESTS} requests used.")
            if not SERPER_ENABLED:
                print("⚠️ Previously exhausted quota detected — Serper remains DISABLED.")
        except (json.JSONDecodeError, ValueError, OSError) as e:
            print(f"Could not read existing usage file ({e}); starting from 0.")
    else:
        print("No existing serper_usage.json found — starting from 0.")


def save_serper_usage():
    """Persist current usage counter to disk. Thread-safe snapshot read."""
    with serper_lock:
        used = serper_requests_used
        enabled = SERPER_ENABLED
    payload = {
        "limit": SERPER_MAX_REQUESTS,
        "used": used,
        "remaining": max(SERPER_MAX_REQUESTS - used, 0),
        "enabled": enabled,
        "last_updated": datetime.now(timezone.utc).isoformat(),
    }
    tmp_path = SERPER_USAGE_FILE.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(payload, f, indent=2)
    tmp_path.replace(SERPER_USAGE_FILE)  # atomic-ish write


load_serper_usage()
save_serper_usage()


No existing serper_usage.json found — starting from 0.


## 4 & 6 & 11 & 12. Serper Request Wrapper — thread-safe, capped, retried, logged

In [5]:
# ============================================================
# 4. THREAD SAFETY  /  6. FALLBACK LOGIC  /  11. ERROR HANDLING
# ============================================================

def _reserve_serper_slot():
    """
    Atomically check-and-increment the request counter.
    Returns True if a slot was reserved (i.e. this caller may proceed to send
    ONE request), False if the quota is exhausted.
    This is the only place that increments serper_requests_used, and it does so
    BEFORE the network call, under the lock, so two threads can never both see
    2499 and both proceed.
    """
    global serper_requests_used, SERPER_ENABLED

    with serper_lock:
        if not SERPER_ENABLED or serper_requests_used >= SERPER_MAX_REQUESTS:
            SERPER_ENABLED = False
            return False
        serper_requests_used += 1
        used_now = serper_requests_used
        if used_now >= SERPER_MAX_REQUESTS:
            SERPER_ENABLED = False

    if used_now % 250 == 0 or used_now == SERPER_MAX_REQUESTS:
        print(f"Serper requests: {used_now} / {SERPER_MAX_REQUESTS}  "
              f"Remaining: {max(SERPER_MAX_REQUESTS - used_now, 0)}")

    if used_now == SERPER_MAX_REQUESTS:
        print("=" * 50)
        print("⚠️ SERPER REQUEST LIMIT REACHED")
        print(f"{SERPER_MAX_REQUESTS} / {SERPER_MAX_REQUESTS} requests used")
        print("Serper fallback has been disabled.")
        print("The pipeline will continue without Serper.")
        print("=" * 50)

    return True


def serper_image_search(query, num_results=10):
    """
    Perform ONE Serper Image Search request (plus up to SERPER_RETRY_LIMIT retries
    for transient errors — retries also consume budget and are capped by it).

    Returns a dict:
        {"status": "ok", "images": [...]}                     on success
        {"status": "quota_limit_reached", "images": []}       quota exhausted
        {"status": "rate_limited", "images": []}               HTTP 429
        {"status": "authentication_error", "images": []}       HTTP 401/403
        {"status": "timeout", "images": []}                    timeout/conn error
        {"status": "error", "images": []}                      other errors
    """
    attempt = 0
    while True:
        if not _reserve_serper_slot():
            return {"status": "quota_limit_reached", "images": []}

        attempt += 1
        try:
            resp = requests.post(
                "https://google.serper.dev/images",
                headers={
                    "X-API-KEY": os.environ.get("SERPER_API_KEY", ""),
                    "Content-Type": "application/json",
                },
                json={"q": query, "num": num_results},
                timeout=SERPER_REQUEST_TIMEOUT,
            )

            if resp.status_code == 429:
                if attempt <= SERPER_RETRY_LIMIT:
                    time.sleep(2 * attempt)
                    continue
                return {"status": "rate_limited", "images": []}

            if resp.status_code in (401, 403):
                # Auth errors won't be fixed by retrying.
                return {"status": "authentication_error", "images": []}

            resp.raise_for_status()
            data = resp.json()
            images = [
                {"url": img.get("imageUrl"), "title": img.get("title", "")}
                for img in data.get("images", [])
                if img.get("imageUrl")
            ]
            return {"status": "ok", "images": images}

        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            if attempt <= SERPER_RETRY_LIMIT:
                time.sleep(2 * attempt)
                continue
            return {"status": "timeout", "images": []}

        except requests.exceptions.RequestException as e:
            print(f"Serper request error (non-retryable): {e}")
            return {"status": "error", "images": []}


## 12. Optional API Key Test (consumes 1 request — off by default)

In [6]:
# ============================================================
# 12. API KEY VALIDATION (optional, costs 1 request)
# ============================================================
TEST_SERPER_API = False  # set True to spend 1 request verifying the key works

def test_serper_api():
    result = serper_image_search("test product packaging", num_results=1)
    if result["status"] == "ok":
        print("Serper API test: SUCCESS")
    else:
        print(f"Serper API test: FAILED ({result['status']})")
    with serper_lock:
        used = serper_requests_used
    print(f"Requests used: {used} / {SERPER_MAX_REQUESTS}")
    return result

if TEST_SERPER_API:
    test_serper_api()
else:
    print("TEST_SERPER_API is False — skipping test request (no budget consumed).")


TEST_SERPER_API is False — skipping test request (no budget consumed).


## 5. Sociolla Direct Image Extraction

Scrapes the product's own Sociolla page for packaging images. This never touches the Serper budget and always runs first for every product, regardless of quota state.

In [7]:
# ============================================================
# 5. SOCIOLLA DIRECT EXTRACTION (does not touch Serper budget)
# ============================================================
import re
from bs4 import BeautifulSoup

SOCIOLLA_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
}

def sociolla_direct_extraction(product_url, timeout=15):
    """
    Fetch a Sociolla product page and pull candidate packaging image URLs
    from it (product gallery / og:image / img tags containing the product
    media CDN path). Returns a list of image URLs (may be empty).
    This is a LOCAL/direct request to Sociolla — it never counts against
    the Serper budget.
    """
    try:
        resp = requests.get(product_url, headers=SOCIOLLA_HEADERS, timeout=timeout)
        resp.raise_for_status()
    except requests.exceptions.RequestException:
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    urls = set()

    # og:image meta tag
    og = soup.find("meta", property="og:image")
    if og and og.get("content"):
        urls.add(og["content"])

    # Product gallery <img> tags — Sociolla product media typically lives
    # under a recognizable CDN path.
    for img in soup.find_all("img"):
        src = img.get("data-src") or img.get("src")
        if src and re.search(r"(product|media|cloudinary|sociolla)", src, re.I):
            if src.startswith("//"):
                src = "https:" + src
            urls.add(src)

    return list(urls)


## Local Image Processing — download, validate, deduplicate

Runs for every image found, whether sourced from Sociolla directly or from Serper. Never counts against the Serper budget.

In [8]:
# ============================================================
# LOCAL IMAGE PROCESSING (downloads, validation, dedup — no API budget cost)
# ============================================================
seen_hashes_lock = Lock()
seen_image_hashes = set()

def download_and_validate_image(image_url, dest_path, min_dim=200, timeout=15):
    """
    Download an image, validate it opens correctly and meets a minimum
    resolution, and dedupe via content hash. Returns True if the image was
    saved, False otherwise. Purely local/network-to-image-host work — never
    touches the Serper counter.
    """
    try:
        resp = requests.get(image_url, timeout=timeout)
        resp.raise_for_status()
        content = resp.content

        img_hash = hashlib.sha256(content).hexdigest()
        with seen_hashes_lock:
            if img_hash in seen_image_hashes:
                return False  # duplicate, skip
            seen_image_hashes.add(img_hash)

        img = Image.open(io.BytesIO(content))
        img.verify()  # validates it's a real image
        img = Image.open(io.BytesIO(content))  # reopen after verify()
        if img.width < min_dim or img.height < min_dim:
            return False

        dest_path.parent.mkdir(parents=True, exist_ok=True)
        with open(dest_path, "wb") as f:
            f.write(content)
        return True

    except Exception:
        return False


## 9 & 10. Per-Product Pipeline

Sociolla first → Serper only if insufficient images → skip gracefully if quota exhausted. Exactly **one** Serper request per product, regardless of how many image types are missing (front/back/official/etc all come from that single request's results).

In [9]:
# ============================================================
# 9. PER-PRODUCT PROCESSING  /  10. SINGLE SERPER QUERY PER PRODUCT
# ============================================================

def process_product(row):
    """
    row: dict with at least brand_name, product_name, product_id, url
    Returns a metadata dict describing what was collected for this product.
    """
    brand = str(row.get("brand_name", "")).strip()
    name = str(row.get("product_name", "")).strip()
    product_id = str(row.get("product_id", "")).strip()
    product_url = str(row.get("url", "")).strip()

    product_dir = IMAGES_DIR / f"{brand}_{product_id}"
    saved_images = []

    # ---- Step 1: Sociolla direct extraction --------------------------
    direct_urls = sociolla_direct_extraction(product_url) if product_url else []
    for i, img_url in enumerate(direct_urls):
        dest = product_dir / f"sociolla_{i}.jpg"
        if download_and_validate_image(img_url, dest):
            saved_images.append(str(dest))

    result = {
        "brand_name": brand,
        "product_name": name,
        "product_id": product_id,
        "url": product_url,
        "images_from_sociolla": len(saved_images),
        "images_from_serper": 0,
        "total_images": len(saved_images),
        "serper_status": "not_needed",
    }

    # ---- Step 2/3/4: Serper fallback, only if needed ------------------
    if len(saved_images) >= MIN_IMAGES_PER_PRODUCT:
        return result  # enough images already — never call Serper

    if not SERPER_ENABLED:
        result["serper_status"] = "quota_limit_reached"
        return result

    query = f"{brand} {name} official product packaging"
    serper_result = serper_image_search(query, num_results=10)  # ONE request
    result["serper_status"] = serper_result["status"]

    if serper_result["status"] == "ok":
        for i, img in enumerate(serper_result["images"]):
            dest = product_dir / f"serper_{i}.jpg"
            if download_and_validate_image(img["url"], dest):
                saved_images.append(str(dest))
        result["images_from_serper"] = len(saved_images) - result["images_from_sociolla"]

    result["total_images"] = len(saved_images)
    return result


## Load Product Dataset

In [10]:
df_products = pd.read_csv(PRODUCTS_CSV)
print(f"Loaded {len(df_products)} products across {df_products['brand_name'].nunique()} brands.")
df_products[["brand_name", "product_name", "product_id", "url"]].head()


Loaded 7636 products across 321 brands.


,brand_name,product_name,product_id,url
0,796_3ce,MULTI EYE COLOR PALETTE,97802,https://www.sociolla.com/eyeshadow/69460-phan-...
1,796_3ce,VELVET LIP TINT,97810,https://www.sociolla.com/lip-cream/69468-son-k...
2,796_3ce,LIP COLOR,97822,https://www.sociolla.com/lip-matte/69480-son-t...
3,796_3ce,MINI MULTI EYE COLOR PALETTE,97833,https://www.sociolla.com/eyeshadow/69491-phan-...
4,796_3ce,FACE BLUSH,97801,https://www.sociolla.com/blush/69459-phan-ma-h...


## 17. Test Mode

Run on a small sample first (default 20 products) to verify the whole pipeline — API key, counter, Sociolla extraction, Serper fallback, persistence, and graceful shutdown behavior — before committing to all 7,636 products.

In [16]:
# ============================================================
# 17. TEST MODE
# ============================================================
TEST_MODE = False
TEST_PRODUCTS = 20

MAX_WORKERS = 8  # ThreadPoolExecutor size


## 13 & 3. Status Display Before Processing

In [17]:
# ============================================================
# 13. DISPLAY STATUS (before processing)
# ============================================================
with serper_lock:
    _used = serper_requests_used
    _enabled = SERPER_ENABLED

print("Serper API:")
print("AVAILABLE" if _enabled else "DISABLED (quota already exhausted)")
print()
print("Serper request budget:")
print(SERPER_MAX_REQUESTS)
print()
print("Requests already used:")
print(_used)
print()
print("Remaining:")
print(max(SERPER_MAX_REQUESTS - _used, 0))
print()
print(f"Mode: {'TEST_MODE (' + str(TEST_PRODUCTS) + ' products)' if TEST_MODE else 'FULL RUN (' + str(len(df_products)) + ' products)'}")


Serper API:
AVAILABLE

Serper request budget:
2500

Requests already used:
20

Remaining:
2480

Mode: FULL RUN (7636 products)


## Run the Pipeline

Sociolla direct extraction always continues, image validation/dedup/metadata writing always continue, even after the Serper quota is exhausted — only the Serper fallback itself is disabled at that point.

In [18]:
# ============================================================
# 3 / 5 / 6: RUN — Sociolla continues regardless of Serper quota state
# ============================================================

products_to_process = (
    df_products.head(TEST_PRODUCTS) if TEST_MODE else df_products
).to_dict(orient="records")

results = []
missing_images_rows = []
processed_count = 0
results_lock = Lock()

def _handle_result(res):
    global processed_count
    with results_lock:
        results.append(res)
        processed_count += 1
        if res["total_images"] == 0:
            missing_images_rows.append(res)
        if processed_count % SERPER_SAVE_EVERY_N_PRODUCTS == 0:
            save_serper_usage()
            print(f"Processed {processed_count} / {len(products_to_process)} products "
                  f"(images so far: {sum(r['total_images'] for r in results)})")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_product, row): row for row in products_to_process}
    for future in as_completed(futures):
        try:
            res = future.result()
        except Exception as e:
            row = futures[future]
            res = {
                "brand_name": row.get("brand_name", ""),
                "product_name": row.get("product_name", ""),
                "product_id": row.get("product_id", ""),
                "url": row.get("url", ""),
                "images_from_sociolla": 0,
                "images_from_serper": 0,
                "total_images": 0,
                "serper_status": f"processing_error: {e}",
            }
        _handle_result(res)

# Final checkpoint save
save_serper_usage()
print("Done. Final usage checkpoint saved.")


Processed 25 / 7636 products (images so far: 49)
Processed 50 / 7636 products (images so far: 278)
Processed 75 / 7636 products (images so far: 460)
Processed 100 / 7636 products (images so far: 629)
Processed 125 / 7636 products (images so far: 748)
Processed 150 / 7636 products (images so far: 882)
Processed 175 / 7636 products (images so far: 1062)
Processed 200 / 7636 products (images so far: 1203)
Serper requests: 250 / 2500  Remaining: 2250
Processed 225 / 7636 products (images so far: 1365)
Processed 250 / 7636 products (images so far: 1528)
Processed 275 / 7636 products (images so far: 1679)
Processed 300 / 7636 products (images so far: 1805)
Processed 325 / 7636 products (images so far: 2020)
Processed 350 / 7636 products (images so far: 2206)
Processed 375 / 7636 products (images so far: 2342)
Processed 400 / 7636 products (images so far: 2464)
Processed 425 / 7636 products (images so far: 2595)
Processed 450 / 7636 products (images so far: 2755)
Serper requests: 500 / 2500  

## Write Metadata Outputs

In [19]:
# ============================================================
# Write missing_images.csv and full results metadata
# ============================================================
results_df = pd.DataFrame(results)
results_path = METADATA_DIR / ("test_results.csv" if TEST_MODE else "collection_results.csv")
results_df.to_csv(results_path, index=False)
print(f"Saved results metadata to {results_path}")

if missing_images_rows:
    missing_df = pd.DataFrame(missing_images_rows)
    missing_df.to_csv(MISSING_IMAGES_FILE, index=False)
    print(f"Saved {len(missing_images_rows)} products with no usable image to {MISSING_IMAGES_FILE}")
else:
    print("No products with zero images — missing_images.csv not written.")


Saved results metadata to SafeCart_Dataset\metadata\collection_results.csv
Saved 5230 products with no usable image to SafeCart_Dataset\metadata\missing_images.csv


## 14. Final Summary

In [20]:
# ============================================================
# 14. FINAL SUMMARY
# ============================================================
with serper_lock:
    _used = serper_requests_used
    _enabled = SERPER_ENABLED

total_products = len(results_df)
direct_only = int(((results_df["images_from_sociolla"] > 0) & (results_df["images_from_serper"] == 0)).sum())
used_serper = int((results_df["images_from_serper"] > 0).sum())
skipped_quota = int((results_df["serper_status"] == "quota_limit_reached").sum())
no_image = int((results_df["total_images"] == 0).sum())
total_images = int(results_df["total_images"].sum())

print("TOTAL PRODUCTS:")
print(total_products if TEST_MODE else len(df_products))
print()
print("PRODUCTS WITH DIRECT SOCIOLLA IMAGES:")
print(direct_only)
print()
print("PRODUCTS USING SERPER:")
print(used_serper)
print()
print("SERPER REQUESTS USED:")
print(f"{_used} / {SERPER_MAX_REQUESTS}")
print()
print("SERPER REQUESTS REMAINING:")
print(max(SERPER_MAX_REQUESTS - _used, 0))
print()
print("SERPER ENABLED:")
print("YES" if _enabled else "NO")
print()
print("PRODUCTS SKIPPED BECAUSE SERPER LIMIT WAS REACHED:")
print(skipped_quota)
print()
print("PRODUCTS WITH NO USABLE IMAGE:")
print(no_image)
print()
print("TOTAL IMAGES:")
print(total_images)


TOTAL PRODUCTS:
7636

PRODUCTS WITH DIRECT SOCIOLLA IMAGES:
0

PRODUCTS USING SERPER:
2406

SERPER REQUESTS USED:
2500 / 2500

SERPER REQUESTS REMAINING:
0

SERPER ENABLED:
NO

PRODUCTS SKIPPED BECAUSE SERPER LIMIT WAS REACHED:
5156

PRODUCTS WITH NO USABLE IMAGE:
5230

TOTAL IMAGES:
16375


## Next Step

Once `TEST_MODE` has been verified end-to-end (key accepted, counter increments
correctly, Sociolla extraction works, Serper fallback triggers only when needed, usage
persists to `serper_usage.json`, and the pipeline doesn't crash if Serper is disabled),
set:

```python
TEST_MODE = False
```

and re-run from the **Test Mode** cell onward to process all 7,636 products. Because
usage is checkpointed to disk, you can safely stop and resume this notebook at any time
without ever exceeding the 2,500-request Serper budget.